In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import pandas as pd
import numpy as np
import json, re
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score

In [2]:
class EnsembleWrapper:
    def __init__(self, reg_skl, clf_skl, gated_pt, attn_pt, device):
        self.reg_sk  = reg_skl
        self.clf_sk  = clf_skl
        self.gated   = gated_pt
        self.attn    = attn_pt
        self.device  = device

    def predict_batch(self, X_struct, X_jd, X_text):
        # 1) sklearn part
        X_full = np.hstack([X_struct, X_jd, X_text])
        r_sk = self.reg_sk.predict(X_full)
        p_sk = self.clf_sk.predict_proba(X_full)

        # 2) PyTorch part
        with torch.no_grad():
            s = torch.from_numpy(X_struct).float().to(self.device)
            j = torch.from_numpy(X_jd).    float().to(self.device)
            t = torch.from_numpy(X_text).  float().to(self.device)

            rg, pg_logits = self.gated.predict(s,j,t)
            ra, pa_logits = self.attn. predict(s,j,t)

            rg = rg.cpu().numpy(); ra = ra.cpu().numpy()
            pg = F.softmax(pg_logits,dim=1).cpu().numpy()
            pa = F.softmax(pa_logits,dim=1).cpu().numpy()

        # 3) Soft‐vote
        r_ens = (r_sk + rg + ra) / 3
        p_ens = (p_sk + pg + pa) / 3
        c_ens = np.argmax(p_ens, axis=1)
        return r_ens, c_ens

In [5]:
class InterviewDataset(Dataset):
    def __init__(self, master_csv, jd_csv):
        df = pd.read_csv(master_csv)
        jd = pd.read_csv(jd_csv)
        df = df.merge(jd[['jd_id','jd_text_embedding']], on='jd_id', how='left')
        
        def parse_emb(s):
            try:
                return np.array(json.loads(s))
            except:
                parts = re.split(r'[,\s]+', s.strip().lstrip('[').rstrip(']'))
                return np.array([float(x) for x in parts if x])
        
        def parse_skills(s):
            return np.array(json.loads(s.replace("'", '"')))
        
        df['struct'] = df.apply(lambda r: np.concatenate([parse_skills(r['skills_vector']), 
                                                           [r['segment_count']]]), axis=1)
        df['text']   = df['transcript_embedding'].apply(parse_emb)
        df['jd']     = df['jd_text_embedding'].   apply(parse_emb)
        
        df = df.dropna(subset=['struct','text','jd'])
        self.s = np.vstack(df['struct'].values).astype(np.float32)
        self.t = np.vstack(df['text'].values).  astype(np.float32)
        self.j = np.vstack(df['jd'].values).    astype(np.float32)
        
        self.y_reg = df['Overall_Score'].values.astype(np.float32)
        rec_cols = ['rec_Hire','rec_Consider','rec_Reject']
        self.y_cls = df[rec_cols].values.argmax(axis=1).astype(np.int64)
        
    def __len__(self):
        return len(self.y_reg)
    
    def __getitem__(self, idx):
        return (self.s[idx], self.j[idx], self.t[idx],
                self.y_reg[idx], self.y_cls[idx])

In [14]:
import numpy as np
# 1. Load data and features
ds = InterviewDataset('../../../../data/test-final/FINAL_master.csv', '../../../data/test-final/jds_clean.csv')
X_struct = ds.s.np()
X_jd     = ds.j.numpy()
X_text   = ds.t.numpy()
y_reg    = ds.y_reg.numpy()
y_cls    = ds.y_cls.numpy()

# 2. Load the saved ensemble wrapper
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ensemble = torch.load("../saved/multimodal_ensemble.pth", map_location=device)
ensemble.eval()

AttributeError: 'numpy.ndarray' object has no attribute 'np'